# Search Algorithm Demo — BFS Category Traversal
**CIC6314 Artificial Intelligence | Smart Product Recommendation System**  
**Member 1 (Issye) — Search Algorithm Component**

---

## Overview

This notebook demonstrates the **Breadth-First Search (BFS)** algorithm used to find relevant product categories for a given customer.

### Problem Being Solved
Given a customer's favourite category, find all **semantically related categories** within 2 hops on a similarity graph — so the recommendation engine explores products beyond just the customer's exact preference.

### Algorithm Choice: BFS
BFS is the optimal choice here because:
- It guarantees **closest categories are discovered first** (level-order traversal)
- It finds all reachable nodes within a bounded hop count (`max_hops=2`)
- The similarity graph is unweighted at the traversal level (edges are pruned by threshold, not ranked), making BFS more appropriate than A*

### Two Functions
| Function | Use case | Input |
|---|---|---|
| `find_reachable_categories()` | Returning customer (has purchase history) | favourite_category from user profile |
| `find_popular_categories()` | New customer (cold-start, no history) | eligible categories from rules engine |

## 1. Setup & Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx
from collections import deque

from src.search_module import find_reachable_categories, find_popular_categories

# Load pkl files
with open('../models/category_similarity.pkl', 'rb') as f:
    cat_sim = pickle.load(f)

with open('../models/product_catalogue.pkl', 'rb') as f:
    catalogue = pickle.load(f)

print('✓ category_similarity.pkl loaded:', cat_sim.shape)
print('✓ product_catalogue.pkl loaded:', catalogue.shape)
print('\nCategories:', list(cat_sim.columns))

## 2. The Category Similarity Graph

The 8×8 similarity matrix is derived from **ALS (Alternating Least Squares)** collaborative filtering — categories that are frequently purchased together by the same customers have higher similarity scores.

**Note on scale:** ALS cosine similarity values in this dataset are in the range 0.03–0.19 (not the standard 0–1), because the latent factor space is shaped by purchase sparsity. The BFS threshold of **0.06** is calibrated to this scale.

In [ ]:
# ── Heatmap of the full 8x8 similarity matrix ─────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))

# Short labels for readability
short_labels = [
    'Home\nDecor', 'Kitchen &\nDining', 'Seasonal\n& Gifts', 'Toys &\nGames',
    'Stationery\n& Craft', 'Fashion &\nAccessories', 'Garden &\nOutdoor', 'Food &\nConf.'
]

mask = np.eye(len(cat_sim), dtype=bool)  # hide diagonal (self-similarity)
sns.heatmap(
    cat_sim,
    annot=True, fmt='.3f',
    cmap='YlOrRd',
    mask=mask,
    xticklabels=short_labels,
    yticklabels=short_labels,
    linewidths=0.5,
    ax=ax,
    vmin=0.03, vmax=0.19
)

# Highlight threshold line
ax.set_title('ALS Category Similarity Matrix\n(BFS threshold = 0.06 — edges below this are pruned)', 
             fontsize=13, fontweight='bold', pad=15)

# Add threshold annotation
fig.text(0.92, 0.15, 'BFS edge\nthreshold\n= 0.06', ha='center', fontsize=9,
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.show()

print("\nFashion & Accessories max similarity (excluding self):")
row = cat_sim['Fashion & Accessories'].drop('Fashion & Accessories')
print(f"  Max = {row.max():.4f} → below threshold 0.06 → correctly isolated by BFS")

## 3. Category Graph Visualisation

Categories are **nodes**. An **edge** exists between two categories only if their similarity ≥ 0.06 (the BFS traversal threshold). This is the graph that BFS explores.

In [ ]:
THRESHOLD = 0.06

# Build NetworkX graph
G = nx.Graph()
categories = list(cat_sim.columns)
G.add_nodes_from(categories)

for i, cat_a in enumerate(categories):
    for j, cat_b in enumerate(categories):
        if i < j:
            sim = cat_sim.loc[cat_a, cat_b]
            if sim >= THRESHOLD:
                G.add_edge(cat_a, cat_b, weight=sim)

# Layout
pos = nx.spring_layout(G, seed=42, k=2.5)

fig, ax = plt.subplots(figsize=(12, 8))

edge_weights = [G[u][v]['weight'] for u, v in G.edges()]
edge_widths  = [w * 25 for w in edge_weights]
edge_colors  = [plt.cm.YlOrRd(w / 0.19) for w in edge_weights]

# Isolated nodes (Fashion & Accessories) in different colour
isolated  = [n for n in G.nodes() if G.degree(n) == 0]
connected = [n for n in G.nodes() if G.degree(n) > 0]

nx.draw_networkx_nodes(G, pos, nodelist=connected, node_size=2200,
                       node_color='#4A90D9', alpha=0.9, ax=ax)
nx.draw_networkx_nodes(G, pos, nodelist=isolated, node_size=2200,
                       node_color='#E74C3C', alpha=0.7, ax=ax)
nx.draw_networkx_edges(G, pos, width=edge_widths, edge_color=edge_colors,
                       alpha=0.8, ax=ax)

# Short labels
labels = {c: c.replace(' & ', '\n& ').replace(' (', '\n(') for c in categories}
nx.draw_networkx_labels(G, pos, labels, font_size=8, font_weight='bold', ax=ax)

# Edge weight labels
edge_labels = {(u, v): f'{G[u][v]["weight"]:.3f}' for u, v in G.edges()}
nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=7, ax=ax)

blue_patch = mpatches.Patch(color='#4A90D9', label='Connected (BFS reachable)')
red_patch  = mpatches.Patch(color='#E74C3C', label='Isolated (pruned by threshold)')
ax.legend(handles=[blue_patch, red_patch], loc='upper left', fontsize=10)

ax.set_title(f'Category Similarity Graph (threshold ≥ {THRESHOLD})\nEdge thickness ∝ similarity strength',
             fontsize=13, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

print(f"Total edges in graph: {G.number_of_edges()}")
print(f"Isolated nodes (max sim < {THRESHOLD}): {isolated}")

## 4. BFS Step-by-Step Trace

To demonstrate how BFS works on this graph, we trace each step of the queue expansion for a sample customer whose favourite category is **Home Decor**.

In [ ]:
def bfs_trace(start, eligible, max_hops=2, threshold=0.06):
    """BFS with full step-by-step trace output."""
    print(f"{'='*65}")
    print(f" BFS TRACE: start='{start}', threshold={threshold}, max_hops={max_hops}")
    print(f"{'='*65}")
    print(f" Eligible categories: {eligible}")
    print(f"{'-'*65}")

    visited = {start}
    queue   = deque([(start, 0)])
    result  = [start] if start in eligible else []
    step    = 0

    print(f" INIT  | Queue: [('{start}', hop=0)]")
    if start in eligible:
        print(f"        → '{start}' is eligible → added to result")

    while queue:
        node, hops = queue.popleft()
        step += 1
        print(f"\n STEP {step} | Processing: '{node}' (hop {hops})")

        if hops >= max_hops:
            print(f"        → max_hops reached — skip expanding")
            continue

        row = cat_sim[node].drop(node).sort_values(ascending=False)
        print(f"        Neighbours (sorted by similarity):")

        for neighbour, sim in row.items():
            if sim < threshold:
                print(f"          {neighbour:<28} sim={sim:.4f} → BELOW threshold, stop")
                break
            status = ''
            if neighbour in visited:
                status = '(already visited)'
            else:
                visited.add(neighbour)
                queue.append((neighbour, hops + 1))
                if neighbour in eligible:
                    result.append(neighbour)
                    status = '→ ADDED TO RESULT'
                else:
                    status = '→ queued (not eligible)'
            print(f"          {neighbour:<28} sim={sim:.4f}  {status}")

    print(f"\n{'='*65}")
    print(f" RESULT: {result}")
    print(f"{'='*65}")
    return result


# Sample: returning customer, all categories eligible
ALL_CATEGORIES = list(cat_sim.columns)
result = bfs_trace('Home Decor', ALL_CATEGORIES)

## 5. Function 1 — `find_reachable_categories()` (Returning Customers)

Testing all 5 sample user profiles from `src/constants.py`. These use real CustomerIDs from the Online Retail dataset.

In [ ]:
ALL_CATS = list(cat_sim.columns)

# Sample user profiles (from src/constants.py)
sample_profiles = [
    {
        'customer_id': 17850,
        'favourite_category': 'Home Decor',
        'customer_segment': 'High Value',
        'price_range': 'Mid-High',
        'description': 'High-value UK customer, frequent buyer'
    },
    {
        'customer_id': 14688,
        'favourite_category': 'Kitchen & Dining',
        'customer_segment': 'Regular',
        'price_range': 'Mid-Low',
        'description': 'Regular customer, kitchen focus'
    },
    {
        'customer_id': 15311,
        'favourite_category': 'Seasonal & Gifts',
        'customer_segment': 'Occasional',
        'price_range': 'Low',
        'description': 'Seasonal shopper, gift items'
    },
    {
        'customer_id': 16029,
        'favourite_category': 'Food & Confectionery',
        'customer_segment': 'Regular',
        'price_range': 'Low',
        'description': 'Regular buyer, food focus'
    },
    {
        'customer_id': None,
        'favourite_category': 'Fashion & Accessories',
        'customer_segment': 'Regular',
        'price_range': 'Mid-High',
        'description': 'Fashion buyer (isolated category test)'
    }
]

print(f"{'Customer':<12} {'Favourite Category':<25} {'Reachable Categories (BFS)':<60} {'Count'}")
print("-" * 115)

for p in sample_profiles:
    reachable = find_reachable_categories(p, ALL_CATS, max_hops=2, threshold=0.06)
    cid = str(p['customer_id']) if p['customer_id'] else 'N/A'
    fav = p['favourite_category']
    cats_str = ', '.join(reachable)
    print(f"{cid:<12} {fav:<25} {cats_str:<60} {len(reachable)}/8")

print()
print("Note: Fashion & Accessories has max neighbour similarity of 0.059 < 0.06 threshold")
print("      → BFS fallback triggers → returns full eligible list (correct behaviour)")

## 6. BFS Path Visualisation

Visualising which nodes are visited during BFS for a **Home Decor** customer (hop 1 = direct neighbours, hop 2 = neighbours of neighbours).

In [ ]:
def get_bfs_hops(start, max_hops=2, threshold=0.06):
    """Return dict: node → hop distance from start."""
    hop_map = {start: 0}
    visited = {start}
    queue   = deque([(start, 0)])

    while queue:
        node, hops = queue.popleft()
        if hops >= max_hops:
            continue
        row = cat_sim[node].drop(node).sort_values(ascending=False)
        for neighbour, sim in row.items():
            if sim < threshold:
                break
            if neighbour not in visited:
                visited.add(neighbour)
                hop_map[neighbour] = hops + 1
                queue.append((neighbour, hops + 1))

    return hop_map


START = 'Home Decor'
hop_map = get_bfs_hops(START)

colour_map = {
    -1: '#E74C3C',   # isolated (not reached)
     0: '#F39C12',   # start node
     1: '#2ECC71',   # hop 1
     2: '#3498DB',   # hop 2
}

node_colours = []
for n in G.nodes():
    if n == START:
        node_colours.append(colour_map[0])
    elif n in hop_map:
        node_colours.append(colour_map[hop_map[n]])
    else:
        node_colours.append(colour_map[-1])

fig, ax = plt.subplots(figsize=(12, 8))
nx.draw_networkx_nodes(G, pos, node_color=node_colours, node_size=2400, alpha=0.9, ax=ax)
nx.draw_networkx_edges(G, pos, width=2.5, edge_color='#BDC3C7', alpha=0.6, ax=ax)
nx.draw_networkx_labels(G, pos, labels, font_size=8, font_weight='bold', ax=ax)
nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=7, ax=ax)

legend_patches = [
    mpatches.Patch(color=colour_map[0],  label='Start node (Home Decor)'),
    mpatches.Patch(color=colour_map[1],  label='Hop 1 — direct neighbours'),
    mpatches.Patch(color=colour_map[2],  label='Hop 2 — neighbours of neighbours'),
    mpatches.Patch(color=colour_map[-1], label='Not reached (Fashion & Accessories isolated)'),
]
ax.legend(handles=legend_patches, loc='upper left', fontsize=10)
ax.set_title(f'BFS Traversal from "{START}" (max_hops=2, threshold=0.06)\nAll reachable categories coloured by hop distance',
             fontsize=13, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

print("BFS hop distances from Home Decor:")
for node, hop in sorted(hop_map.items(), key=lambda x: x[1]):
    print(f"  Hop {hop}: {node}")
unreached = [n for n in ALL_CATS if n not in hop_map]
print(f"  Not reached: {unreached}")

## 7. Function 2 — `find_popular_categories()` (Cold-Start / New Customers)

When a customer has no purchase history, BFS cannot start (no anchor category). Instead, we rank eligible categories by total **unique buyer count** from the product catalogue.

In [ ]:
# Cold-start: new customer, no history
print("=" * 55)
print(" COLD-START — New Customer (no purchase history)")
print("=" * 55)

# All categories eligible (rules engine hasn't filtered anything)
top3 = find_popular_categories(ALL_CATS, price_range='Mid-Low', top_n=3)

print("\nTop 3 most popular categories (by buyer count):")
for rank, (cat, buyers) in enumerate(top3, 1):
    bar = '█' * (buyers // 500)
    print(f"  #{rank}  {cat:<28}  {buyers:>6,} buyers  {bar}")

# Buyer count per category (full breakdown)
print("\nFull category buyer count breakdown:")
counts = (
    catalogue.groupby('category')['popularity_rank']
    .sum()
    .sort_values(ascending=False)
)
for cat, cnt in counts.items():
    bar = '█' * (cnt // 500)
    print(f"  {cat:<28}  {cnt:>8,}  {bar}")

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
colours = ['#F39C12' if i < 3 else '#BDC3C7' for i in range(len(counts))]
bars = ax.barh(counts.index[::-1], counts.values[::-1], color=colours[::-1], edgecolor='white')
ax.set_xlabel('Total Buyer Count (sum of popularity_rank per category)', fontsize=11)
ax.set_title('Cold-Start Ranking: Product Category Popularity\n(top 3 highlighted in orange)', 
             fontsize=13, fontweight='bold')

for bar, val in zip(bars, counts.values[::-1]):
    ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## 8. Integration Point — How This Connects to the Full Pipeline

This search module sits between the **Rules Engine** (Member 2) and the **ML Model** (Member 3).

In [ ]:
# Simulated integration call — what Member 4's pipeline will look like
print("=" * 60)
print(" SIMULATED PIPELINE (Member 4 will wire this together)")
print("=" * 60)

# --- Step 1: Rules Engine output (Member 2) ---
# apply_rules() returns a filtered list of eligible categories
simulated_eligible = [
    'Home Decor', 'Kitchen & Dining', 'Seasonal & Gifts',
    'Toys & Games', 'Garden & Outdoor'
    # (Fashion & Accessories and Stationery filtered out by rules)
]

# --- Step 2: Returning customer path ---
returning_profile = {
    'customer_id': 17850,
    'favourite_category': 'Home Decor',
    'customer_segment': 'High Value',
    'price_range': 'Mid-High'
}

print("\n[Returning Customer]")
print(f"  Input  → eligible: {simulated_eligible}")
print(f"  Input  → favourite_category: '{returning_profile['favourite_category']}'")

reachable = find_reachable_categories(returning_profile, simulated_eligible)
print(f"  Output → BFS reachable: {reachable}")
print(f"  → Passes {len(reachable)} categories to ML model for product ranking")

# --- Step 3: New customer path ---
print("\n[New Customer — Cold Start]")
print(f"  Input  → eligible: {simulated_eligible}")
print(f"  Input  → price_range: 'Mid-High'")

popular = find_popular_categories(simulated_eligible, price_range='Mid-High', top_n=3)
print(f"  Output → top categories: {popular}")
print(f"  → Recommends top 3 by popularity directly")

print("\n" + "=" * 60)
print(" Pipeline: rules_engine → search_module → ml_model → output")
print("=" * 60)

## 9. Summary

| Aspect | Detail |
|---|---|
| **Algorithm** | Breadth-First Search (BFS) |
| **Graph** | 8×8 category similarity matrix derived from ALS collaborative filtering |
| **Edge threshold** | 0.06 (calibrated to ALS similarity scale of 0.03–0.19) |
| **Max hops** | 2 (balances exploration vs. relevance) |
| **Returning customer** | BFS from favourite category → ordered by closeness |
| **New customer** | Popularity ranking by buyer count — no graph traversal needed |
| **Fashion & Accessories** | Correctly isolated — max similarity 0.059 < threshold → BFS fallback |
| **Time complexity** | O(V + E) where V=8, E≤28 — constant time for this fixed graph |

**Why BFS over DFS or A\*?**
- **BFS** is optimal for finding all nodes within a bounded hop count — closest categories discovered first
- **DFS** would not guarantee shortest-path discovery, potentially returning distant categories before close ones  
- **A\*** requires a meaningful heuristic function — in this undirected similarity graph there is no destination node, making A* inapplicable

The implementation is in `src/search_module.py`.